# Social Reasoning benchmark: clustered primary analysis

The analysis samples complete question groups using a fixed seed. The primary and sole metric is full-sequence loss on each `chosen` response, averaged over the four records within each question. Model comparisons use paired question-level inference and Holm correction. The `rejected` field is retained for provenance but is not scored as a separate metric.

Power planning assumes a standardized paired question-level effect of `d_z = 0.20`, 80% power, and a conservative Bonferroni planning alpha for six pairwise comparisons. The computed sample size is printed when the notebook runs.


In [ ]:
import hashlib
import json
import math
import warnings
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
from scipy import stats
from scipy.stats import nct
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("default")

# Reproducibility and design constants
RANDOM_SEED = 42
TARGET_POWER = 0.80
TARGET_PAIRED_EFFECT_DZ = 0.20
FAMILYWISE_ALPHA = 0.05
N_PAIRWISE_COMPARISONS = 6  # four models -> six pairwise comparisons
N_BOOTSTRAP = 5000
MAX_SEQUENCE_LENGTH = 512  # explicit evaluation cap; examples over this cap are NOT silently truncated


def compute_required_question_sample_size(
    effect_size_dz: float = TARGET_PAIRED_EFFECT_DZ,
    alpha: float = FAMILYWISE_ALPHA,
    power: float = TARGET_POWER,
    n_comparisons: int = N_PAIRWISE_COMPARISONS,
    max_n: int = 955,
) -> int:
    """Question clusters required for a paired t-test, using conservative Bonferroni planning.

    effect_size_dz is the standardized paired effect: mean paired difference / SD of
    question-level paired differences. The familywise alpha is divided by the number
    of planned pairwise comparisons for sample-size planning.
    """
    alpha_each = alpha / n_comparisons
    for n in range(2, max_n + 1):
        df = n - 1
        critical_t = stats.t.ppf(1 - alpha_each / 2, df)
        ncp = effect_size_dz * math.sqrt(n)
        achieved_power = nct.sf(critical_t, df, ncp) + nct.cdf(-critical_t, df, ncp)
        if achieved_power >= power:
            return n
    raise ValueError(
        f"Required sample exceeds available questions ({max_n}); revise the design or effect target."
    )


REQUIRED_QUESTIONS = compute_required_question_sample_size()
print(f"Planned sample: {REQUIRED_QUESTIONS} questions × 4 records = {REQUIRED_QUESTIONS * 4} records")
print(f"Planning assumptions: paired d_z={TARGET_PAIRED_EFFECT_DZ}, power={TARGET_POWER:.0%}, "
      f"familywise alpha={FAMILYWISE_ALPHA}, Bonferroni alpha per comparison="
      f"{FAMILYWISE_ALPHA / N_PAIRWISE_COMPARISONS:.5f}; seed={RANDOM_SEED}")


def load_prolific_dataset(filepath: str, n_questions: int = REQUIRED_QUESTIONS, seed: int = RANDOM_SEED):
    """Randomly sample complete question groups, retaining all four records per question."""
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {filepath}")

    grouped = {}
    n_invalid = 0
    with path.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                n_invalid += 1
                continue

            question = str(item.get("question", "")).strip()
            chosen = str(item.get("chosen", "")).strip()
            rejected = str(item.get("rejected", "")).strip()
            if not question or not chosen or not rejected:
                n_invalid += 1
                continue

            qid = hashlib.sha256(question.encode("utf-8")).hexdigest()[:16]
            grouped.setdefault(question, []).append({
                "prompt": question,
                "response": chosen,
                "rejected": rejected,  # retained for data integrity; not scored as an additional metric
                "source": "ProlificAI",
                "question_id": qid,
                "source_line": line_num,
            })

    group_sizes = {q: len(rows) for q, rows in grouped.items()}
    incomplete = {q: n for q, n in group_sizes.items() if n != 4}
    if incomplete:
        examples = list(incomplete.items())[:5]
        raise ValueError(
            f"Expected exactly four records per question; found {len(incomplete)} malformed groups. "
            f"Examples (question prefix, row count): {[(q[:50], n) for q, n in examples]}"
        )
    if len(grouped) < n_questions:
        raise ValueError(f"Requested {n_questions} questions but only found {len(grouped)} complete groups.")

    questions = list(grouped.keys())
    rng = np.random.default_rng(seed)
    selected_questions = rng.choice(len(questions), size=n_questions, replace=False)
    sampled = []
    for idx in selected_questions:
        sampled.extend(grouped[questions[int(idx)]])

    assert len(sampled) == n_questions * 4
    assert len({x["question_id"] for x in sampled}) == n_questions
    print(f"Loaded {len(grouped)} complete question groups; sampled {n_questions} with seed={seed}.")
    print(f"Scoring {len(sampled)} chosen-response records; rejected text is retained but not scored.")
    if n_invalid:
        print(f"Skipped {n_invalid} malformed/empty JSONL records.")
    return sampled


In [ ]:
# Full-sequence loss and question-clustered inference

def _model_context_limit(model, tokenizer):
    candidates = []
    for value in [getattr(model.config, "max_position_embeddings", None),
                  getattr(tokenizer, "model_max_length", None)]:
        if isinstance(value, int) and 0 < value < 1_000_000:
            candidates.append(value)
    return min(candidates) if candidates else MAX_SEQUENCE_LENGTH


def compute_loss(model, tokenizer, prompt, response):
    """Mean causal-LM loss over the tokenized full prompt + chosen response sequence.

    No truncation is performed silently. If the complete sequence exceeds the model's
    supported context length, the function raises an error rather than scoring a prefix.
    """
    model.eval()
    device = model.device
    full_text = prompt + " " + response
    inputs = tokenizer(full_text, return_tensors="pt", truncation=False)
    sequence_length = int(inputs["input_ids"].shape[1])
    context_limit = _model_context_limit(model, tokenizer)
    if sequence_length > context_limit:
        raise ValueError(
            f"Full sequence has {sequence_length} tokens, exceeding context limit {context_limit}. "
            "No truncation was applied; resolve this before comparing models."
        )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return float(outputs.loss.item())


def _cluster_bootstrap_mean_ci(values, n_bootstrap=N_BOOTSTRAP, seed=RANDOM_SEED):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    n = len(values)
    boot = np.empty(n_bootstrap, dtype=float)
    for b in range(n_bootstrap):
        boot[b] = np.mean(values[rng.integers(0, n, size=n)])
    return [float(x) for x in np.percentile(boot, [2.5, 97.5])]


def compute_statistics_on_examples(model, tokenizer, examples: List[Dict], batch_size: int = 4) -> Dict:
    """Score chosen records; aggregate the four record losses within each question first."""
    row_losses = []
    row_question_ids = []
    for example in tqdm(examples, desc="Computing full-sequence losses"):
        row_losses.append(compute_loss(model, tokenizer, example["prompt"], example["response"]))
        row_question_ids.append(example["question_id"])

    by_question = {}
    for qid, loss in zip(row_question_ids, row_losses):
        by_question.setdefault(qid, []).append(loss)
    malformed = {qid: len(v) for qid, v in by_question.items() if len(v) != 4}
    if malformed:
        raise ValueError(f"Expected four scored records per question; found malformed groups: {malformed}")

    question_ids = list(by_question.keys())
    question_scores = np.array([np.mean(by_question[qid]) for qid in question_ids], dtype=float)
    mean_loss = float(np.mean(question_scores))
    std_loss = float(np.std(question_scores, ddof=1))
    sem_loss = float(stats.sem(question_scores))
    t_ci = stats.t.interval(0.95, len(question_scores) - 1, loc=mean_loss, scale=sem_loss)
    bootstrap_ci = _cluster_bootstrap_mean_ci(question_scores)

    # Perplexity is retained only as a descriptive transformation of the native-tokenizer loss.
    return {
        "losses": [float(x) for x in row_losses],
        "row_question_ids": row_question_ids,
        "question_ids": question_ids,
        "question_scores": question_scores.tolist(),
        "n_records": len(row_losses),
        "n_questions": len(question_scores),
        "mean_loss": mean_loss,
        "std_loss": std_loss,
        "sem_loss": sem_loss,
        "ci_95_lower": float(t_ci[0]),
        "ci_95_upper": float(t_ci[1]),
        "bootstrap_ci_95": bootstrap_ci,
        "perplexity": float(np.exp(mean_loss)),
    }


def _holm_adjust(p_values):
    """Holm step-down adjusted p-values, returned in input order."""
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adjusted_sorted = np.empty(m, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        candidate = min(1.0, (m - rank) * p[idx])
        running_max = max(running_max, candidate)
        adjusted_sorted[rank] = running_max
    adjusted = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        adjusted[idx] = adjusted_sorted[rank]
    return adjusted.tolist()


def compute_pairwise_statistics(results_by_model: Dict[str, Dict]) -> Dict:
    """Paired tests and bootstrap CIs over question-level means; Holm over all model pairs."""
    model_names = list(results_by_model.keys())
    comparisons = {}
    pending = []
    for i, model1 in enumerate(model_names):
        for model2 in model_names[i + 1:]:
            r1, r2 = results_by_model[model1], results_by_model[model2]
            ids1, ids2 = r1["question_ids"], r2["question_ids"]
            if ids1 != ids2:
                map1 = dict(zip(ids1, r1["question_scores"]))
                map2 = dict(zip(ids2, r2["question_scores"]))
                common = sorted(set(map1) & set(map2))
                if len(common) != len(ids1) or len(common) != len(ids2):
                    raise ValueError("Models were not evaluated on the same question clusters.")
                x = np.array([map1[q] for q in common], dtype=float)
                y = np.array([map2[q] for q in common], dtype=float)
            else:
                x = np.asarray(r1["question_scores"], dtype=float)
                y = np.asarray(r2["question_scores"], dtype=float)
            diff = x - y
            t_stat, p_value = stats.ttest_rel(x, y)
            sd_diff = float(np.std(diff, ddof=1))
            dz = float(np.mean(diff) / sd_diff) if sd_diff > 0 else float("nan")

            rng = np.random.default_rng(RANDOM_SEED + i * 101 + model_names.index(model2))
            boot = np.empty(N_BOOTSTRAP, dtype=float)
            n = len(diff)
            for b in range(N_BOOTSTRAP):
                ix = rng.integers(0, n, size=n)
                boot[b] = np.mean(diff[ix])
            ci = [float(v) for v in np.percentile(boot, [2.5, 97.5])]

            key = f"{model1.split('/')[-1]}_vs_{model2.split('/')[-1]}"
            pending.append((key, {
                "model1": model1.split("/")[-1],
                "model2": model2.split("/")[-1],
                "n_questions": n,
                "mean_loss_diff": float(np.mean(diff)),  # model1 minus model2; lower loss is better
                "t_statistic": float(t_stat),
                "p_value": float(p_value),
                "cohens_dz_paired": dz,
                "bootstrap_ci_95": ci,
                "test_type": "paired_ttest_on_question_means",
            }))

    adjusted = _holm_adjust([item[1]["p_value"] for item in pending])
    for (key, result), p_adj in zip(pending, adjusted):
        result["p_value_holm"] = float(p_adj)
        result["significant_holm_05"] = bool(p_adj < FAMILYWISE_ALPHA)
        comparisons[key] = result
    return comparisons


def generate_visualizations(results: Dict, output_dir: str = "."):
    """Create loss-CI and paired-effect-size figures; omit the p-value heatmap."""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
    except ImportError:
        print("Matplotlib/seaborn unavailable; skipping visualizations.")
        return

    output_dir = Path(output_dir)
    models = list(results["model_results"].keys())
    names = [m.split("/")[-1] for m in models]
    means = [results["model_results"][m]["mean_loss"] for m in models]
    cis = [results["model_results"][m]["bootstrap_ci_95"] for m in models]
    lower = [means[i] - cis[i][0] for i in range(len(models))]
    upper = [cis[i][1] - means[i] for i in range(len(models))]

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(names))
    ax.bar(x, means, yerr=[lower, upper], capsize=5, alpha=0.7)
    ax.set_xlabel("Model")
    ax.set_ylabel("Full-sequence loss (lower is better)")
    ax.set_title("Social Reasoning loss by model (question-bootstrap 95% CI)")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=35, ha="right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / "loss_comparison_with_ci.png", dpi=300, bbox_inches="tight")
    plt.close()

    effect = np.zeros((len(models), len(models)))
    comparisons = results.get("pairwise_statistics", {})
    for comp in comparisons.values():
        i = names.index(comp["model1"])
        j = names.index(comp["model2"])
        effect[i, j] = comp["cohens_dz_paired"]
        effect[j, i] = -comp["cohens_dz_paired"]
    plt.figure(figsize=(10, 8))
    sns.heatmap(effect, annot=True, fmt=".3f", cmap="RdBu_r", xticklabels=names,
                yticklabels=names, center=0, cbar_kws={"label": "Paired Cohen's dz"})
    plt.title("Paired effect sizes (row-model loss minus column-model loss)")
    plt.tight_layout()
    plt.savefig(output_dir / "effect_size_heatmap.png", dpi=300, bbox_inches="tight")
    plt.close()


In [ ]:
# Main experiment

import os
# Stable result keys are kept separate from local checkpoint paths.
_models = ["Qwen3-8B", "Qwen3-8B-Base", "llama-3.1-8b-base", "llama-3.1-8B-instruct"]
_model_paths = {
    "Qwen3-8B": os.environ.get("QWEN3_INSTRUCT_PATH", "Qwen3-8B"),
    "Qwen3-8B-Base": os.environ.get("QWEN3_BASE_PATH", "Qwen3-8B-Base"),
    "llama-3.1-8b-base": os.environ.get("LLAMA31_BASE_PATH", "llama-3.1-8b-base"),
    "llama-3.1-8B-instruct": os.environ.get("LLAMA31_INSTRUCT_PATH", "llama-3.1-8B-instruct"),
}

_output = "results/social_reasoning/primary_clustered_307/social_reasoning_results.json"
_prolific_file = "data/social_reasoning/ProlificAI.jsonl"
_visualize = True
_batch_size = 4

examples = load_prolific_dataset(_prolific_file, n_questions=REQUIRED_QUESTIONS, seed=RANDOM_SEED)
question_ids_sampled = list(dict.fromkeys(e["question_id"] for e in examples))

all_results = {
    "metadata": {
        "dataset": _prolific_file,
        "total_dataset_questions_expected": 955,
        "n_questions_sampled": REQUIRED_QUESTIONS,
        "records_per_question": 4,
        "n_records_sampled": len(examples),
        "sampling_unit": "distinct question text; complete four-record groups",
        "sampling_method": "simple random sample without replacement",
        "random_seed": RANDOM_SEED,
        "primary_metric": "mean full-sequence causal-LM loss on chosen records",
        "secondary_metrics": [],
        "rejected_field_policy": "retained in sampled records for provenance; not scored",
        "power_analysis": {
            "target_power": TARGET_POWER,
            "target_standardized_paired_effect_dz": TARGET_PAIRED_EFFECT_DZ,
            "familywise_alpha": FAMILYWISE_ALPHA,
            "number_of_pairwise_comparisons": N_PAIRWISE_COMPARISONS,
            "planning_correction": "Bonferroni",
            "alpha_per_comparison": FAMILYWISE_ALPHA / N_PAIRWISE_COMPARISONS,
            "required_question_clusters": REQUIRED_QUESTIONS,
        },
        "inference": "paired t-tests on question-level means; paired question bootstrap CIs; Holm-adjusted p-values",
        "sequence_policy": f"no truncation; sequences exceeding model context limit raise an error (nominal cap {MAX_SEQUENCE_LENGTH} is not silently applied)",
    },
    "model_results": {},
}

for model_path in _models:
    print(f"\n{'=' * 80}\nEvaluating: {model_path}\n{'=' * 80}")
    checkpoint_path = _model_paths[model_path]
    model = AutoModelForCausalLM.from_pretrained(
        checkpoint_path, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, trust_remote_code=True)
    model.eval()
    try:
        result = compute_statistics_on_examples(model, tokenizer, examples, _batch_size)
        all_results["model_results"][model_path] = result
        print(f"Questions: {result['n_questions']}; records: {result['n_records']}")
        print(f"Mean full-sequence loss: {result['mean_loss']:.5f}")
        print(f"Question-bootstrap 95% CI: {result['bootstrap_ci_95']}")
        print(f"Native-tokenizer PPL (descriptive only): {result['perplexity']:.3f}")
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if len(all_results["model_results"]) < 2:
    raise RuntimeError("Fewer than two models were evaluated; pairwise inference cannot be performed.")

all_results["pairwise_statistics"] = compute_pairwise_statistics(all_results["model_results"])
print("\nPairwise comparisons (negative Δ means model 1 has lower loss):")
for name, comp in all_results["pairwise_statistics"].items():
    print(f"\n{name}")
    print(f"  Δ loss (model 1 - model 2): {comp['mean_loss_diff']:+.5f}")
    print(f"  95% paired question-bootstrap CI: [{comp['bootstrap_ci_95'][0]:.5f}, {comp['bootstrap_ci_95'][1]:.5f}]")
    print(f"  paired Cohen's dz: {comp['cohens_dz_paired']:+.3f}")
    print(f"  raw p={comp['p_value']:.6g}; Holm-adjusted p={comp['p_value_holm']:.6g}; "
          f"significant after Holm={comp['significant_holm_05']}")

with open(_output, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print(f"\nSaved results to {_output}")
if _visualize:
    generate_visualizations(all_results, Path(_output).parent)
    print("Saved loss-CI and paired-effect-size figures (no p-value heatmap).")
